# FitMyResume Phase 9 Notebook Demo

This notebook demonstrates the end-to-end FitMyResume workflow for the CS455 project: provide a resume and job description, then display a fit score, explanation, and resume improvement suggestions.

The default demo path uses the saved fine-tuned validation sample in `results/finetuned_qwen_validation_transformers_sample50_outputs.jsonl`, so it can run without a GPU or local vLLM server. Optional live inference notes are included near the end for GPU-backed demos.

## Demo Backend

- **Default:** saved fine-tuned model output from the Transformers/PEFT validation sample.
- **Optional live path:** run `src/run_finetuned_vllm_inference.py` against a vLLM OpenAI-compatible server, or run `src/run_finetuned_transformers_inference.py` with the local LoRA adapter.
- **Fallback reason:** local notebooks often do not have enough GPU memory for Qwen2.5-7B plus LoRA inference, so the saved-output path is the reliable presentation demo.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from textwrap import shorten
from typing import Any

MAX_INPUT_CHARS = 12000
RESULTS_PATH = Path("results/finetuned_qwen_validation_transformers_sample50_outputs.jsonl")
INSTRUCTIONS_PATH = Path("data/instruction_tuning/instruction_tuning_validation.jsonl")


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find project root. Run this notebook from the project checkout.")


PROJECT_ROOT = find_project_root()
RESULTS_FILE = (PROJECT_ROOT / RESULTS_PATH).resolve()
INSTRUCTIONS_FILE = (PROJECT_ROOT / INSTRUCTIONS_PATH).resolve()
print(f"Project root: {PROJECT_ROOT}")
print(f"Demo artifact: {RESULTS_FILE}")
print(f"Instruction inputs: {INSTRUCTIONS_FILE}")

In [ ]:
def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as input_file:
        for line_number, line in enumerate(input_file, start=1):
            if not line.strip():
                continue
            row = json.loads(line)
            if not isinstance(row, dict):
                raise ValueError(f"{path} line {line_number} is not a JSON object")
            rows.append(row)
    return rows


def split_instruction_input(input_text: str) -> tuple[str, str]:
    resume_marker = "RESUME:\n"
    job_marker = "\n\nJOB_DESCRIPTION:\n"
    if resume_marker not in input_text or job_marker not in input_text:
        return input_text.strip(), ""
    resume_text = input_text.split(resume_marker, 1)[1].split(job_marker, 1)[0]
    job_text = input_text.split(job_marker, 1)[1]
    return resume_text.strip(), job_text.strip()


def validate_input_lengths(resume_text: str, job_description: str, max_chars: int = MAX_INPUT_CHARS) -> None:
    resume_chars = len(resume_text)
    job_chars = len(job_description)
    if resume_chars > max_chars:
        raise ValueError(f"Resume is {resume_chars:,} characters; limit is {max_chars:,}.")
    if job_chars > max_chars:
        raise ValueError(f"Job description is {job_chars:,} characters; limit is {max_chars:,}.")


def get_parsed_output(row: dict[str, Any]) -> dict[str, Any]:
    parsed = row.get("parsed_output")
    if isinstance(parsed, dict):
        return parsed

    raw_response = row.get("raw_response", "")
    if isinstance(raw_response, str) and raw_response.strip():
        return json.loads(raw_response)

    raise ValueError("Row does not contain a parsed output or JSON raw response.")


def find_demo_row(rows: list[dict[str, Any]]) -> dict[str, Any]:
    parseable_rows = [row for row in rows if row.get("parse_success") and isinstance(row.get("parsed_output"), dict)]
    if not parseable_rows:
        raise ValueError("No parseable fine-tuned outputs found in the demo artifact.")
    return max(parseable_rows, key=lambda row: int(row.get("parsed_output", {}).get("score", -1)))


def instruction_rows_by_pair_id(rows: list[dict[str, Any]]) -> dict[str, dict[str, Any]]:
    by_pair_id: dict[str, dict[str, Any]] = {}
    for row in rows:
        metadata = row.get("metadata", {})
        if not isinstance(metadata, dict):
            continue
        pair_id = metadata.get("pair_id")
        if isinstance(pair_id, str) and pair_id:
            by_pair_id[pair_id] = row
    return by_pair_id


def get_source_input(output_row: dict[str, Any], instruction_lookup: dict[str, dict[str, Any]]) -> str:
    pair_id = output_row.get("pair_id", "")
    source_row = instruction_lookup.get(str(pair_id))
    if not source_row:
        raise KeyError(f"Could not find instruction input for pair_id={pair_id!r}")
    return str(source_row.get("input", ""))

In [ ]:
def display_fitmyresume_output(output: dict[str, Any]) -> None:
    score = output.get("score", "N/A")
    explanation = output.get("explanation", {})
    suggestions = output.get("resume_suggestions", [])

    print("FIT SCORE")
    print(f"{score}/100")
    print()

    print("MATCHED QUALIFICATIONS")
    for item in explanation.get("matched_qualifications", []):
        print(f"- {item}")
    print()

    print("MISSING OR WEAK QUALIFICATIONS")
    for item in explanation.get("missing_or_weak_qualifications", []):
        print(f"- {item}")
    print()

    print("OVERALL REASONING")
    print(explanation.get("overall_reasoning", ""))
    print()

    print("RESUME SUGGESTIONS")
    for index, suggestion in enumerate(suggestions, start=1):
        if not isinstance(suggestion, dict):
            print(f"{index}. {suggestion}")
            continue
        section = suggestion.get("section", "section")
        action = suggestion.get("action", "revise")
        text = suggestion.get("suggestion", "")
        evidence = suggestion.get("evidence_from_resume", "")
        requirement = suggestion.get("job_requirement_addressed", "")
        print(f"{index}. [{section} / {action}] {text}")
        if evidence:
            print(f"   Evidence: {evidence}")
        if requirement:
            print(f"   Requirement addressed: {requirement}")

## Polished Presentation Example

This cell loads a representative parseable fine-tuned output, extracts the original resume and job description, checks input length, and displays the final user-facing sections.

In [ ]:
demo_rows = read_jsonl(RESULTS_FILE)
instruction_rows = read_jsonl(INSTRUCTIONS_FILE)
instruction_lookup = instruction_rows_by_pair_id(instruction_rows)
demo_row = find_demo_row(demo_rows)
demo_output = get_parsed_output(demo_row)
resume_text, job_description = split_instruction_input(get_source_input(demo_row, instruction_lookup))
validate_input_lengths(resume_text, job_description)

print(f"Pair ID: {demo_row.get('pair_id', '')}")
print(f"Backend: {demo_row.get('serving_backend', 'saved output')}")
print(f"Resume preview: {shorten(resume_text, width=350, placeholder=' ...')}")
print(f"Job preview: {shorten(job_description, width=350, placeholder=' ...')}")

In [ ]:
display_fitmyresume_output(demo_output)

## Try Another Saved Example

Change `example_index` to inspect a different saved fine-tuned output. The notebook filters to parseable rows first.

In [ ]:
parseable_rows = [row for row in demo_rows if row.get("parse_success") and isinstance(row.get("parsed_output"), dict)]
example_index = 0

selected_row = parseable_rows[example_index]
selected_resume, selected_job = split_instruction_input(get_source_input(selected_row, instruction_lookup))
validate_input_lengths(selected_resume, selected_job)

print(f"Pair ID: {selected_row.get('pair_id', '')}")
print(f"Resume chars: {len(selected_resume):,}")
print(f"Job description chars: {len(selected_job):,}")
display_fitmyresume_output(get_parsed_output(selected_row))

## Optional Live Inference

Use these commands only in a GPU runtime with the base model and LoRA adapter available. They are not required for the saved-output notebook demo.

### vLLM server path

Start a vLLM OpenAI-compatible server using the selected model or LoRA configuration, then run:

```powershell
python src/run_finetuned_vllm_inference.py `
  --input data/instruction_tuning/instruction_tuning_validation.jsonl `
  --output results/demo_vllm_outputs.jsonl `
  --model fitmyresume `
  --base-url http://localhost:8000/v1 `
  --limit 3
```

### Transformers/PEFT fallback path

```powershell
python src/run_finetuned_transformers_inference.py `
  --input data/instruction_tuning/instruction_tuning_validation.jsonl `
  --output results/demo_transformers_outputs.jsonl `
  --adapter-path models/qwen25-7b-fitmyresume-lora-v2/final `
  --load-in-4bit `
  --limit 3
```

After generating a new output file, set `RESULTS_PATH` at the top of this notebook to the new JSONL path and rerun the display cells.